In [1]:
!pip install pdfplumber sentence-transformers faiss-cpu rank-bm25 transformers nltk rouge-score accelerate bitsandbytes torch

In [2]:
import pdfplumber

def extract_text(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            content = page.extract_text()
            if content: text += content + "\n"
    return text

raw_text = extract_text("Introduccion_al_la_estadistica.pdf")
print(f"Dataset cargado.")

Dataset cargado.


In [3]:
#chunking
import re

def get_chunks(text, chunk_size=600, overlap=120):
    # Se tomará un patrón para identificar secciones y títulos, para enriquecer el contexto
    pattern = r"\n\d+\.\d+\s+[A-ZÁÉÍÓÚÑ].*"
    sections = re.split(pattern, text)
    titles = re.findall(pattern, text)

    if len(sections) > len(titles):
        titles.insert(0, "Inicio/Prefacio")

    final_chunks = []

    # Al ser un documento de matemáticas excluiremos algunas frases para que no las tome como contexto
    blacklisted_words = ["ejercicio", "pregunta", "responda", "cuestionario", "repaso"]

    for i, content in enumerate(sections):
        title = titles[i].strip()
        # transformamos los saltos de lines en espacios
        clean_sec = re.sub(r'\s+', ' ', content)

        start = 0
        while start < len(clean_sec):
            end = start + chunk_size
            chunk_text = clean_sec[start:end].strip()

            # Filtros para mejorar el contexto
            # verificamos si es una pregunta
            has_questions = "?" in chunk_text or "¿" in chunk_text

            # Excluimos frases incluidas en blacklisted_words
            has_bad_keywords = any(word in chunk_text.lower() for word in blacklisted_words)

            # Validamos los filtros y tamaño
            if len(chunk_text) > 150 and not has_questions and not has_bad_keywords:
                final_chunks.append({
                    "title": title,
                    "content": chunk_text
                })

            # Desplazamiento de ventana con overlap
            start += (chunk_size - overlap)

    return final_chunks

chunks_data = get_chunks(raw_text)
texts = [c["content"] for c in chunks_data]

print(f"Total de chunks válidos: {len(chunks_data)}")
if len(chunks_data) > 0:
    print(f"Ejemplo de chunk limpio:\n{chunks_data[0]['content'][:150]}...")

Total de chunks válidos: 2002
Ejemplo de chunk limpio:
rupos y asociaciones 891 E Hojas de soluciones 897 F Oraciones, símbolos y fórmulas matemáticas 901 G Notas para las calculadoras TI-83, 83+, 84 y 84+...


In [4]:
# embeddings
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

embeddings = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
print(f"Embeddings generados en: {device}")

c:\Users\tinoc\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2182.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings generados en: cpu


In [5]:
# FAISS HNSW - se configura FAISS usando IndexHNSWFlat
import faiss

dim = embeddings.shape[1]
index = faiss.IndexHNSWFlat(dim, 32)
index.hnsw.efConstruction = 160
index.add(embeddings.astype('float32'))
print("Índice FAISS HNSW configurado.")

Índice FAISS HNSW configurado.


In [6]:
#RETRIEVAL
from rank_bm25 import BM25Okapi
import numpy as np
# Inicializar BM25
tokenized_corpus = [doc.lower().split() for doc in texts]
bm25 = BM25Okapi(tokenized_corpus)

def hybrid_retrieval(query, k=10):
    # FAISS
    q_emb = embedder.encode([query], normalize_embeddings=True)
    _, sem_idx = index.search(q_emb.astype('float32'), k)

    # BM25
    bm25_scores = bm25.get_scores(query.lower().split())
    lex_idx = np.argsort(bm25_scores)[-k:][::-1]

    # Combinación Híbrida
    combined_indices = list(set(sem_idx[0].tolist() + lex_idx.tolist()))
    return combined_indices

In [7]:
# RERANKER
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

def rerank(query, indices, top_k=3):
    pairs = [(query, texts[i]) for i in indices]
    scores = reranker.predict(pairs)
    ranked_indices = [indices[i] for i in np.argsort(scores)[::-1]]
    return ranked_indices[:top_k]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4486.39it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# LLM ACTUALIZADO (Qwen 1.5 - 4B)
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model_name = "Qwen/Qwen1.5-4B-Chat"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

qwen_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

def generate_qwen(query, context_indices):
    # Usamos los 2 contextos para darle margen de búsqueda
    primary_contexts = "\n".join([texts[i] for i in context_indices[:2]])

    # PROMPT OPTIMIZADO
    prompt = f"""<|im_start|>system
Eres un experto en estadística y para poder dar una respuesta ten en cuenta las siguientes reglas
REGLAS:
1. NO parafrasees.
2. Usa las palabras exactas del TEXTO.
3. Si no está literal, responde: "Información no encontrada".<|im_end|>
<|im_start|>user
TEXTO: {primary_contexts}
PREGUNTA: {query}<|im_end|>
<|im_start|>assistant
"""

    # Generación sin aleatoriedad (temperature=0)
    output = qwen_pipe(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        temperature=0
    )

    response = output[0]['generated_text'].split("assistant\n")[-1].strip()
    return response

Loading weights: 100%|██████████| 483/483 [00:07<00:00, 66.13it/s] 
Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [10]:
# Hallucination & Grounding
import nltk
from sentence_transformers import util

def process_response_with_metrics(query, raw_ans, context_indices):
    # Citation per sentence
    sentences = nltk.sent_tokenize(raw_ans)
    cited_answer = ""
    for sent in sentences:
        s_emb = embedder.encode(sent)
        c_embs = embedder.encode([texts[i] for i in context_indices])
        best = context_indices[util.cos_sim(s_emb, c_embs).argmax()]
        cited_answer += f"{sent} [Fuente: {chunks_data[best]['title']}] "

    # Grounding
    full_ctx_text = " ".join([texts[i] for i in context_indices])
    g_score = util.cos_sim(embedder.encode(raw_ans), embedder.encode(full_ctx_text)).item()

    # Hallucination
    if g_score < 0.45:
        return "ALERTA: Respuesta con baja evidencia.", g_score

    return cited_answer, g_score

In [11]:
# Para pruebas y verificación del modelo ingresamos 6 querys
# BLEU
import pandas as pd
import re
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import sent_tokenize

def run_extended_benchmark():
    test_queries = [
        "¿Qué es la media aritmética?",
        "¿Qué es el teorema del límite central?",
        "¿Qué es la desviación típica?",
        "¿Qué es la regresión lineal?",
        "¿Qué es la probabilidad condicional?",
        "¿Qué es un experimento binomial?"
    ]

    results = []
    print(f" Benchmark para 6 consultas")

    for i, query in enumerate(test_queries):
        # Ejecución del pipeline
        raw_indices = hybrid_retrieval(query)
        best_indices = rerank(query, raw_indices)
        raw_answer = generate_qwen(query, best_indices)

        # Comparamos la respuesta contra oraciones individuales del contexto
        main_context = texts[best_indices[0]]
        context_sentences = sent_tokenize(main_context)

        # Similitud semántica para encontrar la oración "target" en el libro
        ans_emb = embedder.encode(raw_answer)
        ctx_sent_embs = embedder.encode(context_sentences)
        best_sent_idx = util.cos_sim(ans_emb, ctx_sent_embs).argmax().item()
        target_reference = context_sentences[best_sent_idx]

        # Limpieza para normalización léxica
        def clean_final(txt):
            t = txt.lower()
            t = re.sub(r'[^a-z0-9áéíóúñ\s]', '', t)
            return t.split()

        ref_tokens = [clean_final(target_reference)]
        cand_tokens = clean_final(raw_answer)

        # 4. Cálculo de métricas
        # ROUGE-L
        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        rouge_l = scorer.score(target_reference, raw_answer)['rougeL'].fmeasure

        # BLEU
        smoothie = SmoothingFunction().method4
        bleu_score = sentence_bleu(ref_tokens, cand_tokens, weights=(0.8, 0.2, 0, 0), smoothing_function=smoothie)

        # Grounding
        reference_full_text = " ".join([texts[idx] for idx in best_indices])
        full_ctx_emb = embedder.encode(reference_full_text)
        g_score = util.cos_sim(ans_emb, full_ctx_emb).item()

        results.append({
            "N°": i+1,
            "Query": query[:35] + "...",
            "Grounding": g_score,
            "ROUGE-L": rouge_l,
            "BLEU": bleu_score
        })
        print(f"✔ Procesada consulta {i+1}/6", end="\r")

    # Generación de reporte final en tabla
    df = pd.DataFrame(results)
    print("\n\n" + "="*85)
    print("REPORTE DE RENDIMIENTO RAG: POR QUERY")
    print("="*85)
    print(df.to_string(index=False))
    print("-" * 85)
    print(f"RESUMEN GLOBAL DEL SISTEMA:")
    print(f"  > Grounding (Precisión Semántica):  {df['Grounding'].mean():.4f}")
    print(f"  > ROUGE-L (Capacidad de Síntesis): {df['ROUGE-L'].mean():.4f}")
    print(f"  > BLEU (Fidelidad Terminológica):  {df['BLEU'].mean():.4f} ")
    print("="*85)

# Ejecutar el benchmark masivo
run_extended_benchmark()

 Benchmark para 6 consultas


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✔ Procesada consulta 6/6

REPORTE DE RENDIMIENTO RAG: POR QUERY
 N°                                  Query  Grounding  ROUGE-L     BLEU
  1        ¿Qué es la media aritmética?...   0.705162 0.256410 0.219730
  2 ¿Qué es el teorema del límite centr...   0.730414 0.973684 0.963170
  3       ¿Qué es la desviación típica?...   0.715107 0.428571 0.425199
  4        ¿Qué es la regresión lineal?...   0.701983 0.227273 0.314373
  5 ¿Qué es la probabilidad condicional...   0.678478 0.338462 0.229466
  6    ¿Qué es un experimento binomial?...   0.711961 0.818182 0.749101
-------------------------------------------------------------------------------------
RESUMEN GLOBAL DEL SISTEMA:
  > Grounding (Precisión Semántica):  0.7072
  > ROUGE-L (Capacidad de Síntesis): 0.5071
  > BLEU (Fidelidad Terminológica):  0.4835 


In [12]:
#Demo
def evaluate_query(query):
    print(f"\n Pregunta: {query}\n")

    # Retrieval + Rerank
    try:
        raw_indices = hybrid_retrieval(query, k=10)
    except NameError:
        raise Exception(" Error: 'hybrid_retrieval' no está definido.")

    best_indices = rerank(query, raw_indices, top_k=3)

    # Generación
    raw_answer = generate_qwen(query, best_indices)

    print(" Respuesta :\n")
    print(raw_answer, "\n")

    # Selección de referencia
    from nltk.tokenize import sent_tokenize
    from sentence_transformers import util
    import re

    main_context = texts[best_indices[0]]
    context_sentences = sent_tokenize(main_context)

    ans_emb = embedder.encode(raw_answer)
    ctx_sent_embs = embedder.encode(context_sentences)

    best_sent_idx = util.cos_sim(ans_emb, ctx_sent_embs).argmax().item()
    target_reference = context_sentences[best_sent_idx]

    print("Referencia:\n")
    print(target_reference, "\n")

    # Limpieza
    def clean(txt):
        t = txt.lower()
        t = re.sub(r'[^a-z0-9áéíóúñ\s]', '', t)
        return t.split()

    ref_tokens = [clean(target_reference)]
    cand_tokens = clean(raw_answer)

    # BLEU
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    smoothie = SmoothingFunction().method4

    bleu = sentence_bleu(
        ref_tokens,
        cand_tokens,
        weights=(0.8, 0.2, 0, 0),
        smoothing_function=smoothie
    )

    # ROUGE-L
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_l = scorer.score(target_reference, raw_answer)['rougeL'].fmeasure

    # Grounding
    full_context = " ".join([texts[i] for i in best_indices])
    full_ctx_emb = embedder.encode(full_context)

    grounding = util.cos_sim(ans_emb, full_ctx_emb).item()

    # Resultados

    return {
        "query": query,
        "bleu": bleu,
        "rouge_l": rouge_l,
        "grounding": grounding,
        "answer": raw_answer
    }

In [16]:
#Ejecución de DEMO
evaluate_query("¿Qué es un experimento binomial?")


 Pregunta: ¿Qué es un experimento binomial?



Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Respuesta :

Un experimento binomial es aquel en el que se cuenta el número de aciertos en uno o más ensayos de Bernoulli. 

Referencia:

Un experimento binomial se produce cuando se cuenta el número de aciertos en uno o más ensayos de Bernoulli. 



{'query': '¿Qué es un experimento binomial?',
 'bleu': 0.7491006231418208,
 'rouge_l': 0.8181818181818182,
 'grounding': 0.7119612693786621,
 'answer': 'Un experimento binomial es aquel en el que se cuenta el número de aciertos en uno o más ensayos de Bernoulli.'}